# Sliding-Window Pairwise Electrode Correlation

Compute a time-resolved `n_ch x n_ch` zero-lag Pearson correlation matrix for each of BLT, P1, and P2 (500 ms gap only), on a shared time axis, then compare.

- Signal per condition: trial-averaged evoked.
- Window: `WINDOW_MS` ms, step `STEP_MS` ms.
- Output per condition: `(n_windows, n_ch, n_ch)` + window-center times.

## A. Setup & config

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.stats.correlation_tools as correlation_tools
import statsmodels.tsa.stattools as stattools
import scipy.stats as stats
#from scikit_rmt.ensemble.spectral_law import MarchenkoPasturDistribution

from utils import load_eeg_data
from utils import plot_all_channels_by_trial
from connectivity import (
    select_p2_500ms,
    sliding_corr_from_epochs,
    summary_metrics,
    lagged_corr_2d,
    sliding_lagged_diff_corr_from_epochs
)

SUBJECT = 5
TMIN, TMAX = None, None       # native epoch window (must match across files)
WINDOW_MS = 200
STEP_MS = 50
FREQ_BAND = None              # e.g. 'alpha' / 'beta' for Cell G

## B. Load the three conditions on a common time axis

In [ ]:
blt = load_eeg_data('BLT', SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
bla = load_eeg_data('BLA', SUBJECT, tmin = TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p1_first_10  = load_eeg_data('P1',  SUBJECT, trials = list(range(10)) , tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p1_last_10 = load_eeg_data('P1',  SUBJECT, trials = list(range(-10,0)) , tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p1 = load_eeg_data('P1',  SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p2_all = load_eeg_data('P2', SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p2 = select_p2_500ms(p2_all)
p2_first_10 = load_eeg_data('P2',  SUBJECT, trials = list(range(10)) , tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)
p2_last_10 = load_eeg_data('P2',  SUBJECT, trials = list(range(-10,0)) , tmin=TMIN, tmax=TMAX, freq_band=FREQ_BAND, normalize=True)

assert np.allclose(blt.times, p1.times), 'BLT and P1 time axes differ'
assert np.allclose(p1.times, p2.times), 'P1 and P2 time axes differ'

# Channel-set alignment: intersect across files so the NxN matrices line up.
common_ch = [c for c in blt.ch_names if c in p1.ch_names and c in p2.ch_names]
for ep in (blt, p1, p2):
    ep.pick_channels(common_ch, ordered=True)

print(f'n_channels (common): {len(common_ch)}')
print(f'n_times: {len(blt.times)}   sfreq: {blt.info["sfreq"]} Hz')
print(f'trials BLT/P1/P2_500ms: {len(blt)}/{len(p1)}/{len(p2)}')

## C. Compute sliding-window correlation tensors

In [ ]:
conds = {'BLT': blt, 'P1': p1, 'P2_500ms': p2}
results = {}
for name, ep in conds.items():
    corr, centers, ch_names = sliding_corr_from_epochs(ep, WINDOW_MS, STEP_MS)
    results[name] = {'corr': corr, 'centers': centers, 'ch_names': ch_names}
    print(f'{name}: corr {corr.shape}  centers [{centers[0]:.3f}s .. {centers[-1]:.3f}s]')

# Sanity: all conditions share the same window grid.
ref_centers = results['BLT']['centers']
for name in ('P1', 'P2_500ms'):
    assert np.allclose(results[name]['centers'], ref_centers), f'{name} window centers differ'

## D. Time-course summary per condition

`mean_abs_r` = average `|r|` across unique channel pairs; `density(0.5)` = fraction of pairs with `|r| > 0.5`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for name, res in results.items():
    m = summary_metrics(res['corr'])
    axes[0].plot(res['centers'], m['mean_abs_r'], label=name)
    axes[1].plot(res['centers'], m['density'], label=name)
axes[0].set_ylabel('mean |r|')
axes[1].set_ylabel('density (|r|>0.5)')
axes[1].set_xlabel('time (s)')
for ax in axes:
    ax.axvline(0, color='k', lw=0.5, alpha=0.4)
    ax.legend(loc='best', fontsize=9)
fig.suptitle(f'Sliding-window connectivity (window={WINDOW_MS} ms, step={STEP_MS} ms)')
plt.tight_layout()
plt.show()

## E. Heatmap grid at key time points

Pick a few window indices and show the full `n_ch x n_ch` matrix for each condition.

In [ ]:
centers = results['BLT']['centers']
n_windows = len(centers)
# Pick 5 evenly spaced windows across the epoch.
pick_idx = np.linspace(0, n_windows - 1, 5, dtype=int)
pick_times = centers[pick_idx]

cond_names = list(results.keys())
fig, axes = plt.subplots(
    len(cond_names), len(pick_idx),
    figsize=(3.0 * len(pick_idx), 3.0 * len(cond_names)),
    squeeze=False,
)
for i, name in enumerate(cond_names):
    corr = results[name]['corr']
    for j, w in enumerate(pick_idx):
        ax = axes[i, j]
        im = ax.imshow(corr[w], vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_title(f't={pick_times[j]:+.2f}s', fontsize=10)
        if j == 0:
            ax.set_ylabel(name, fontsize=11)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='Pearson r')
fig.suptitle('Correlation matrices at selected windows')
plt.show()

## F. Condition-difference heatmaps

P1 - BLT isolates the effect of adding the auditory cue (with a fixed 500 ms gap). P2_500ms - P1 isolates paradigm/block differences when the audio->tactile interval is matched.

In [ ]:
diffs = {
    'P1 - BLT': results['P1']['corr'] - results['BLT']['corr'],
    'P2_500ms - P1': results['P2_500ms']['corr'] - results['P1']['corr'],
}
# Symmetric color range per diff for fair reading.
fig, axes = plt.subplots(
    len(diffs), len(pick_idx),
    figsize=(3.0 * len(pick_idx), 3.0 * len(diffs)),
    squeeze=False,
)
for i, (name, d) in enumerate(diffs.items()):
    vmax = np.nanpercentile(np.abs(d), 99)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    for j, w in enumerate(pick_idx):
        ax = axes[i, j]
        im = ax.imshow(d[w], vmin=-vmax, vmax=vmax, cmap='RdBu_r', aspect='auto')
        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_title(f't={pick_times[j]:+.2f}s', fontsize=10)
        if j == 0:
            ax.set_ylabel(name, fontsize=11)
    fig.colorbar(im, ax=axes[i, :].tolist(), shrink=0.7, label=f'\u0394r (vmax={vmax:.2f})')
fig.suptitle('Condition differences in correlation structure')
plt.show()

## G. (Optional) Per-band repeat

Re-run B-F with `FREQ_BAND = 'alpha'` or `'beta'` to check whether the dynamics are band-specific.

In [ ]:
frequencies = ['alpha', 'beta']
for freq in frequencies:
    blt = load_eeg_data('BLT', SUBJECT, tmin=TMIN, tmax=TMAX, freq_band = freq, normalize=True)
    p1  = load_eeg_data('P1',  SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=freq, normalize=True)
    p2_all = load_eeg_data('P2', SUBJECT, tmin=TMIN, tmax=TMAX, freq_band=freq, normalize=True)
    p2 = select_p2_500ms(p2_all)
    
    assert np.allclose(blt.times, p1.times), 'BLT and P1 time axes differ'
    assert np.allclose(p1.times, p2.times), 'P1 and P2 time axes differ'
    
    # Channel-set alignment: intersect across files so the NxN matrices line up.
    common_ch = [c for c in blt.ch_names if c in p1.ch_names and c in p2.ch_names]
    for ep in (blt, p1, p2):
        ep.pick_channels(common_ch, ordered=True)
        
    centers = results['BLT']['centers']
    n_windows = len(centers)
    # Pick 5 evenly spaced windows across the epoch.
    pick_idx = np.linspace(0, n_windows - 1, 5, dtype=int)
    pick_times = centers[pick_idx]
    
    cond_names = list(results.keys())
    fig, axes = plt.subplots(
        len(cond_names), len(pick_idx),
        figsize=(3.0 * len(pick_idx), 3.0 * len(cond_names)),
        squeeze=False,
    )
    for i, name in enumerate(cond_names):
        corr = results[name]['corr']
        for j, w in enumerate(pick_idx):
            ax = axes[i, j]
            im = ax.imshow(corr[w], vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
            ax.set_xticks([])
            ax.set_yticks([])
            if i == 0:
                ax.set_title(f't={pick_times[j]:+.2f}s', fontsize=10)
            if j == 0:
                ax.set_ylabel(name, fontsize=11)
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='Pearson r')
    fig.suptitle('Correlation matrices at selected windows')
    plt.show()
    
    diffs = {
        'P1 - BLT': results['P1']['corr'] - results['BLT']['corr'],
        'P2_500ms - P1': results['P2_500ms']['corr'] - results['P1']['corr'],
    }
    # Symmetric color range per diff for fair reading.
    fig, axes = plt.subplots(
        len(diffs), len(pick_idx),
        figsize=(3.0 * len(pick_idx), 3.0 * len(diffs)),
        squeeze=False,
    )
    for i, (name, d) in enumerate(diffs.items()):
        vmax = np.nanpercentile(np.abs(d), 99)
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        for j, w in enumerate(pick_idx):
            ax = axes[i, j]
            im = ax.imshow(d[w], vmin=-vmax, vmax=vmax, cmap='RdBu_r', aspect='auto')
            ax.set_xticks([])
            ax.set_yticks([])
            if i == 0:
                ax.set_title(f't={pick_times[j]:+.2f}s', fontsize=10)
            if j == 0:
                ax.set_ylabel(name, fontsize=11)
        fig.colorbar(im, ax=axes[i, :].tolist(), shrink=0.7, label=f'\u0394r (vmax={vmax:.2f})')
    fig.suptitle('Condition differences in correlation structure')
    plt.show()
    

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import io
from PIL import Image

centers = results['BLT']['centers']
cond_names = list(results.keys())

frame_images = []

for w in range(len(centers)):
    fig, axes = plt.subplots(
        len(cond_names), 1,
        figsize=(3.5, 3.0 * len(cond_names)),
        squeeze=False,
        constrained_layout=True,   # ← replaces tight_layout
    )

    for i, name in enumerate(cond_names):
        corr = results[name]['corr']
        ax = axes[i, 0]
        im = ax.imshow(corr[w], vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_ylabel(name, fontsize=11)

    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='Pearson r')
    fig.suptitle(f'Correlation matrices — t={centers[w]:+.2f}s', fontsize=12)
    # NO plt.tight_layout() here

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100)
    buf.seek(0)
    frame_images.append(np.array(Image.open(buf)))
    plt.close(fig)

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML
from matplotlib import rc
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 2**128
rc('animation', html='jshtml')

fig, ax = plt.subplots(figsize=(12, 12))
ax.axis("off")
im = ax.imshow(frame_images[0])

def update(frame):
    im.set_array(frame_images[frame])
    return im,

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(frame_images),
    interval=200,
    blit=True,
    repeat=True
)

plt.close(fig)
HTML(ani.to_jshtml())

## Test for Linear Lag

observe difference in correlation between lagged/unlagged, if you see uniform increase in correlation, should hint toward linear lag

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

LAGS_SEC = np.linspace(-0.05, 0.05, 81)

conds = {'P1': p1, "P1 First 10": p1_first_10, "P1 Last 10": p1_last_10,
         'P2 First 10': p2_first_10, 'P2 Last 10': p2_last_10, "BLT": blt, "BLA": bla, "P2 All": p2_all}
results = {}
for name, ep in conds.items():
    diff_corr, raw_corr, centers, ch_names = sliding_lagged_diff_corr_from_epochs(
        ep, WINDOW_MS, STEP_MS, LAGS_SEC
    )
    results[name] = {
        'corr':     diff_corr,
        'raw_corr': raw_corr,
        'centers':  centers,
        'ch_names': ch_names,
    }
    print(f'{name}: corr {diff_corr.shape}  centers [{centers[0]:.3f}s .. {centers[-1]:.3f}s]')

cond_names = list(results.keys())

%matplotlib inline

all_data = np.concatenate([results[n]['corr'] for n in cond_names])
clim     = np.nanpercentile(np.abs(all_data), 99)

lag_slider = widgets.SelectionSlider(
    options=[(f'{l:+.3f}s', i) for i, l in enumerate(LAGS_SEC)],
    description='Lag:',
    layout=widgets.Layout(width='700px'),
)

out = widgets.Output()


def redraw(epoch: str, change=None):
    diff_corr = results[epoch]['corr']      # (n_windows, n_lags, n_ch, n_ch)
    raw_corr  = results[epoch]['raw_corr']  # (n_windows, n_lags, n_ch, n_ch)
    ch_names  = results[epoch]['ch_names']

    lag_idx = lag_slider.value
    lag     = LAGS_SEC[lag_idx]

    # ── Global optimal lag ────────────────────────────────────────────────────
    temporal_variance   = raw_corr.var(axis=0)            # (n_lags, n_ch, n_ch)
    global_variance     = temporal_variance.mean(axis=(1, 2))  # (n_lags,)
    best_lag_global_idx = global_variance.argmin()
    best_lag_global_sec = LAGS_SEC[best_lag_global_idx]

    # ── Per-pair optimal lag ──────────────────────────────────────────────────
    best_lag_idx_pp  = temporal_variance.argmin(axis=0)   # (n_ch, n_ch)
    best_lag_sec_pp  = LAGS_SEC[best_lag_idx_pp]

    raw_corr_mean    = raw_corr.mean(axis=0)              # (n_lags, n_ch, n_ch)
    best_lag_corr_pp = np.take_along_axis(
        raw_corr_mean, best_lag_idx_pp[np.newaxis], axis=0
    ).squeeze(0)
    best_lag_var_pp  = np.take_along_axis(
        temporal_variance, best_lag_idx_pp[np.newaxis], axis=0
    ).squeeze(0)

    threshold     = np.percentile(best_lag_var_pp, 50)
    graph_weights = np.where(best_lag_var_pp < threshold, best_lag_corr_pp, 0)

    # Time-averaged correlation at global optimal lag
    corr_at_global = raw_corr_mean[best_lag_global_idx]   # (n_ch, n_ch)

    # ── Nearest correlation matrix (Higham) ───────────────────────────────────
    def symmetrize_matrix(A):
        return (A + A.T) / 2

    def project_to_positive_semidefinite(A):
        eigenvalues, eigenvectors = np.linalg.eigh(A)
        A_psd = (eigenvectors * np.maximum(eigenvalues, 0)).dot(eigenvectors.T)
        return symmetrize_matrix(A_psd)

    def nearest_correlation_matrix(A, tol=1e-8, max_iterations=100):
        X = symmetrize_matrix(A)
        correction_matrix = np.zeros_like(X)
        for _ in range(max_iterations):
            X_old = X.copy()
            residual = X - correction_matrix
            X = project_to_positive_semidefinite(residual)
            correction_matrix = X - residual
            np.fill_diagonal(X, 1)
            if np.linalg.norm(X - X_old, 'fro') / np.linalg.norm(X, 'fro') < tol:
                break
        return X

    higham_corr = nearest_correlation_matrix(best_lag_corr_pp)

    lambda_max = 4
    x          = np.linspace(-3, 8, 500)
    def mpdist(lambd):
        return np.sqrt((lambda_max - lambd) * lambd) / (2 * np.pi * lambd)

    raw_eig           = np.linalg.eig(corr_at_global)
    normal_eig        = np.linalg.eig(best_lag_corr_pp)
    lag_corrected_eig = np.linalg.eig(higham_corr)
    corrected_corr    = correlation_tools.corr_clipped(corr_at_global, threshold=lambda_max)

    # ── Plot ──────────────────────────────────────────────────────────────────
    with out:
        clear_output(wait=True)
        fig, axes = plt.subplots(3, 3, figsize=(18, 12), constrained_layout=True)

        # [0,0] Δ Corr averaged over all time at selected lag
        mean_diff_at_lag = results['P1']['corr'][:, lag_idx].mean(axis=0)
        im0 = axes[0, 0].imshow(mean_diff_at_lag, vmin=-clim, vmax=clim,
                                 cmap='RdBu_r', aspect='equal')
        axes[0, 0].set_title(f'Δ Corr (time-avg)  |  lag={lag:+.3f}s')
        fig.colorbar(im0, ax=axes[0, 0], label='Δ Pearson r', shrink=0.8)

        # [0,1] Variance curve across lags
        axes[0, 1].plot(LAGS_SEC * 1000, global_variance, color='steelblue')
        axes[0, 1].axvline(best_lag_global_sec * 1000, color='r', linestyle='--',
                           label=f'optimal = {best_lag_global_sec*1000:+.1f}ms')
        axes[0, 1].axvline(lag * 1000, color='orange', linestyle=':',
                           label=f'selected = {lag*1000:+.1f}ms')
        axes[0, 1].set_xlabel('Lag (ms)')
        axes[0, 1].set_ylabel('Mean temporal variance')
        axes[0, 1].set_title('Stability across lags (global)')
        axes[0, 1].legend(fontsize=9)

        # [0,2] Correlation at global optimal lag (time-averaged)
        vmax_g = np.abs(corr_at_global).max()
        im2 = axes[0, 2].imshow(corr_at_global, cmap='RdBu_r', aspect='equal',
                                  vmin=-vmax_g, vmax=vmax_g)
        axes[0, 2].set_title(f'Corr at global optimal lag ({best_lag_global_sec*1000:+.1f}ms, time-avg)')
        fig.colorbar(im2, ax=axes[0, 2], label='Pearson r', shrink=0.8)

        # [1,0] Per-pair optimal lag map
        im3 = axes[1, 0].imshow(best_lag_sec_pp, cmap='RdBu_r', aspect='equal',
                                  vmin=-LAGS_SEC.max(), vmax=LAGS_SEC.max())
        axes[1, 0].set_title('Per-pair optimal lag (s)')
        fig.colorbar(im3, ax=axes[1, 0], label='lag (s)', shrink=0.8)

        # [1,1] Correlation at per-pair optimal lag (time-averaged)
        vmax_pp = np.abs(best_lag_corr_pp).max()
        im4 = axes[1, 1].imshow(best_lag_corr_pp, cmap='RdBu_r', aspect='equal',
                                  vmin=-vmax_pp, vmax=vmax_pp)
        axes[1, 1].set_title('Corr at per-pair optimal lag (time-avg)')
        fig.colorbar(im4, ax=axes[1, 1], label='Pearson r', shrink=0.8)

        # [1,2] Graph weights (thresholded)
        vmax_w = np.abs(graph_weights).max() or 1
        im5 = axes[1, 2].imshow(graph_weights, cmap='RdBu_r', aspect='equal',
                                  vmin=-vmax_w, vmax=vmax_w)
        axes[1, 2].set_title('Graph weights (thresholded to 50th percentile)')
        fig.colorbar(im5, ax=axes[1, 2], label='Pearson r', shrink=0.8)

        # [2,0] Eigenvalue distributions
        axes[2, 0].hist(normal_eig[0],        bins=25, density=True, label='Lag-corrected')
        axes[2, 0].hist(lag_corrected_eig[0], bins=25, density=True, label='Higham')
        axes[2, 0].hist(raw_eig[0],           bins=25, density=True, label='Raw')
        axes[2, 0].plot(x, mpdist(x), label='MP distribution')
        axes[2, 0].legend()
        axes[2, 0].set_title('Eigenvalue distributions')

        # [2,1] Corrected graph weights
        vmax_w = np.abs(corrected_corr).max() or 1
        im6 = axes[2, 1].imshow(corrected_corr, cmap='RdBu_r', aspect='equal',
                                  vmin=-vmax_w, vmax=vmax_w)
        axes[2, 1].set_title('Corrected Graph Weights(without Higham)')
        fig.colorbar(im6, ax=axes[2, 1], label='Pearson r', shrink=0.8)

        # [2,2] Higham correlation matrix
        vmax_h = np.abs(higham_corr).max()
        im7 = axes[2, 2].imshow(higham_corr, cmap='RdBu_r', aspect='equal',
                                  vmin=-vmax_h, vmax=vmax_h)
        axes[2, 2].set_title("Correlation matrix (Higham's Algorithm)")
        fig.colorbar(im7, ax=axes[2, 2], label='Pearson r', shrink=0.8)

        for ax in [axes[0, 0], axes[0, 2], axes[1, 0], axes[1, 1], axes[1, 2]]:
            ax.set_xticks(range(len(ch_names)))
            ax.set_yticks(range(len(ch_names)))
            ax.set_xticklabels(ch_names, rotation=90, fontsize=6)
            ax.set_yticklabels(ch_names, fontsize=6)

        fig.suptitle(f'Time-averaged  |  lag = {lag:+.3f}s  |  epoch = {epoch}', fontsize=12)
        plt.show()

    return corrected_corr, higham_corr, corr_at_global, best_lag_sec_pp, best_lag_corr_pp


lag_slider.observe(lambda change: redraw('P1', change), names='value')
display(lag_slider, out)
corrected, higham = redraw('P1')

In [ ]:
epfjq,wepof, global_corr, opt_lag, lag_applied = redraw("P1")

In [ ]:
print(opt_lag.flatten().max())

In [ ]:
print(1792-.05*512)

In [ ]:
zeroes = 0
zero_channels = []
for i in range(len(opt_lag)):
    for j in range(len(opt_lag)):
        if np.round(opt_lag[i][j], 10) == 0.0:
            zero_channels.append([i,j])
            zeroes += 1
print(zeroes)
print(zero_channels)

In [ ]:
print(3.5*512)

In [ ]:
p1_avg = np.mean(p1.get_data(), axis = 0)
print(np.shape(p1_avg))

In [ ]:
new_corr = []
signal_noise_ratio = []
lags = []
time = len(p1.times)
for i in range(len(opt_lag)):
    x_corr = []
    x_ratio = []
    xlags = []
    for j in range(len(opt_lag)):
        if i == j:
            x_corr.append(1.0)
            x_ratio.append(1.0)
            xlags.append(0.0)
            continue
        lag = opt_lag[i][j]
        lag_samples = int(np.ceil(np.round(lag*512,1)))
        lagged_graph = lagged_corr_2d(p1_avg,lag_samples)

        eig = np.linalg.eigh(lagged_graph)
        eigenvalues = eig[0]
        eigenvectors = eig[1]
        for k in eigenvalues:
            if k < 0:
                print("negative eigen",k, i,j, lag, lag_samples, eigenvalues.max())
                break
        lag_var = np.mean(eigenvalues)
       #print(lag_var)
        # lag 1 by half lag 1 by half in other direction

        lambda_plus = lag_var * 4
        noise_mean = eigenvalues[eigenvalues <= lambda_plus].mean()
        clipped = np.where(eigenvalues > lambda_plus, eigenvalues, 0)
        clean_lagged = eigenvectors @ np.diag(clipped) @ eigenvectors.T
        
        clean_corr = clean_lagged[i][j]
        ratio = np.sign(clean_corr)*clean_corr/lag_applied[i][j]
        x_ratio.append(ratio)
        x_corr.append(clean_corr)
        xlags.append(lag_samples)
    signal_noise_ratio.append(x_ratio)
    new_corr.append(x_corr)
    lags.append(xlags)
print(np.shape(lags))
np.savetxt("lagamount.csv", lags, delimiter = ",")

In [ ]:
fig, ax = plt.subplots(2,2)
fig.set_size_inches(10,10)
im1 = ax[0,0].imshow(new_corr, cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)
ax[0,0].set_title('Correlation from global lag applied')

im2 = ax[0,1].imshow(lag_applied, cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)
ax[0,1].set_title('Correlation with individual lags applied')

im3 = ax[1,0].imshow(signal_noise_ratio, cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)
ax[1,0].set_title('Signal Noise Ratio')

im4 = ax[1,1].imshow(np.subtract(lag_applied, new_corr), cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)
ax[1,1].set_title('Difference')


In [ ]:
lagged_graph = lagged_corr_2d(p1_avg,30)
fig, ax = plt.subplots()
fig.set_size_inches(5,5)
im1 = ax.imshow(lagged_graph, cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)
ax.set_title('Weights for raw data')


for m in range(len(lagged_graph)):
    for n in range(len(lagged_graph)):
        if m > n:
            lagged_graph[m][n] = -lagged_graph[m][n]
eig = np.linalg.eigh(lagged_graph)
eigenvalues = eig[0]
eigenvectors = eig[1]


count = 0
for k in eigenvalues:
    if k < 0:
        count += 1
        print(k)
        print("negative eigen")
print(count)

In [ ]:
corrected = correlation_tools.corr_clipped(lag_applied, threshold=4)

fig, ax = plt.subplots()
fig.set_size_inches(5,5)
im1 = ax.imshow(corrected, cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)
ax.set_title('Weights for raw data')

In [ ]:
dummy = lag_applied.copy()

eig = np.linalg.eigh(dummy)
eigenvalues = eig[0]
eigenvectors = eig[1]
negative, mp, good = 0,0,0
for i in eigenvalues:
    if i < 0:
        negative += 1
    elif 0 <= i <= 4:
        mp += 1
    else:
        good += 1
print(negative,mp,good)
lag_var = np.mean(eigenvalues)

lambda_plus = lag_var * 4
noise_mean = eigenvalues[eigenvalues <= lambda_plus].mean()
clipped = np.where(eigenvalues > lambda_plus, eigenvalues, noise_mean)
clean_lagged = eigenvectors @ np.diag(clipped) @ eigenvectors.T

lag_diff = np.subtract(lag_applied, clean_lagged)

fig, ax = plt.subplots(2,2)
fig.set_size_inches(10,10)
im1 = ax[0,0].imshow(lag_applied, cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)
ax[0,0].set_title('Per-pair optimal lag (s)')

im2 = ax[0,1].imshow(clean_lagged, cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)
ax[0,1].set_title('Per-pair optimal lag (s)')

im2 = ax[1,0].imshow(lag_diff, cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)
ax[1,0].set_title('Per-pair optimal lag (s)')
#fig.colorbar(lag_diff, ax=ax[2], label='lag (s)', shrink=0.8)
mean = 0
minimum = 1000000
for i in range(len(clean_lagged)):
    for j in range(len(clean_lagged)):
        if i == j:
            mean += clean_lagged[i][j]/32
            if clean_lagged[i][j] < minimum:
                minimum = clean_lagged[i][j]
print(mean)
print(minimum)
#im3 = ax[1,1].imshow(dummy, cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)
#ax[1,1].set_title('Per-pair optimal lag (s)')

## PLV implementation


In [ ]:
from scipy.signal import filtfilt, butter

def bandpass(data, fmin, fmax, sfreq, order=4):
    """data: (n_epochs, n_ch, n_times)"""
    nyq = sfreq / 2
    b, a = butter(order, [fmin / nyq, fmax / nyq], btype='band')
    return filtfilt(b, a, data, axis=-1)

def hilbert_plv(epochs, fmin=8, fmax=30):
    """
    Compute time-varying PLV across epochs using Hilbert transform.

    Returns
    -------
    plv_matrix : (n_ch, n_ch, n_times)
    times      : (n_times,)
    """
    data  = epochs.get_data()              # (n_epochs, n_ch, n_times)
    sfreq = epochs.info['sfreq']
    n_epochs, n_ch, n_times = data.shape

    # 1. Bandpass filter into band of interest
    filtered = bandpass(data, fmin, fmax, sfreq)  # (n_epochs, n_ch, n_times)

    # 2. Extract instantaneous phase via Hilbert
    from scipy.signal import hilbert
    phase = np.angle(hilbert(filtered, axis=-1))   # (n_epochs, n_ch, n_times)

    # 3. Compute PLV across epochs for every channel pair and time point
    # PLV[i,j,t] = |mean_over_epochs( exp(i * (phase_i - phase_j)) )|
    # Vectorised version of step 3
    phase_i = phase[:, :, np.newaxis, :]   # (n_epochs, n_ch, 1,    n_times)
    phase_j = phase[:, np.newaxis, :, :]   # (n_epochs, 1,    n_ch, n_times)
    plv_matrix = np.abs(np.mean(np.exp(1j * (phase_i - phase_j)), axis=0))
    # → (n_ch, n_ch, n_times)

    return plv_matrix, epochs.times


plv_matrix, times = hilbert_plv(p1, fmin=8, fmax=30)
print(plv_matrix.shape)  # (32, 32, n_times)

In [ ]:
time_slider = widgets.SelectionSlider(
    options=[(f'{t:+.3f}s', i) for i, t in enumerate(plv_centers)],
    description='Time:',
    layout=widgets.Layout(width='700px'),
)

out = widgets.Output()

def redraw(change=None):
    t_idx = time_slider.value
    with out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)
        im = ax.imshow(
            plv_matrix[:, :, t_idx],
            vmin=0, vmax=1,
            cmap='RdBu_r',
            aspect='equal',
        )
        ax.set_xticks(range(n_ch))
        ax.set_yticks(range(n_ch))
        ax.set_xticklabels(ch_names, rotation=90, fontsize=6)
        ax.set_yticklabels(ch_names, fontsize=6)
        fig.colorbar(im, ax=ax, shrink=0.8, label='PLV')
        ax.set_title(f'PLV (8–30 Hz) — t={plv_centers[t_idx]:+.3f}s', fontsize=12)
        plt.show()

time_slider.observe(redraw, names='value')
display(widgets.VBox([time_slider, out]))
redraw()

In [ ]:
import networkx as nx
import numpy as np

frontal = ['FPZ', 'AF7', 'AF8', 'AF3', 'AF4', 'F3', 'FZ', 'F4']
frontocentral = ['FC5', 'FC1', 'FC2', 'FC6']
somatosensory = ['C3', 'Cz', 'C4']
centroparietal = ['CP3', 'CP1', 'CPZ', 'CP2', 'CP4']
temporal = ['T7', 'T8', 'TP7', 'TP8']
parietal_occipital = ['P5', 'P1', 'P2', 'P6', 'POz', 'O1', 'Oz', 'O2']

groups = {"frontal": frontal, "frontocentral": frontocentral, "somatosensory": somatosensory,
          "centroparietal": centroparietal, "temporal": temporal, "parietal_occipital": parietal_occipital}
colordict = {"frontal": "red", "frontocentral": "orange", "somatosensory": "yellow",
             "centroparietal": "green", "temporal": "blue", "parietal_occipital": "purple"}

CH_TO_GROUP = {ch: g for g, chs in groups.items() for ch in chs}


def initialize_graph(epochs):
    graph = nx.Graph()
    for i, ch in enumerate(epochs.ch_names):
        group = CH_TO_GROUP.get(ch, "")
        graph.add_node(i, color=colordict[group], name=ch, group=group)
    return graph


def build_corr_graph(epochs, mask, weights, threshold=0.2, use_abs=True):
    g = initialize_graph(epochs)
    mask = np.array(mask)
    weights = np.array(weights)
    n = len(epochs.ch_names)
    iu, ju = np.triu_indices(n, k=1)
    keep = mask[iu, ju] > threshold
    w = weights[iu[keep], ju[keep]]
    if use_abs:
        w = np.abs(w)
    g.add_weighted_edges_from(zip(iu[keep].tolist(), ju[keep].tolist(), w.tolist()))
    return g


#g = build_corr_graph(p1, higham_corr, corrected_corr, threshold=0.0, use_abs=False)
#colors = [g.nodes[n]['color'] for n in g.nodes]
#nx.draw(g, node_color=colors, with_labels=True)


In [ ]:
initialize_graph(p1)
g_signal_noise = build_corr_graph(p1,mask = signal_noise_ratio,weights = new_corr, threshold = 0.5)
pos = nx.forceatlas2_layout(g_signal_noise)
nx.draw(g_signal_noise, pos = nx.forceatlas2_layout(g_signal_noise),node_color = [g_signal_noise.nodes[n]['color'] for n in g_signal_noise.nodes], with_labels = True)


In [ ]:
from graphs import run_graph


In [ ]:
import plotly.graph_objects as go
def plot_3d_graph(g):
        pos = nx.forceatlas2_layout(g, dim=3)

        edge_x, edge_y, edge_z = [], [], []
        for u, v in g.edges():
            x0, y0, z0 = pos[u]
            x1, y1, z1 = pos[v]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]
            edge_z += [z0, z1, None]

        edge_trace = go.Scatter3d(
            x=edge_x, y=edge_y, z=edge_z,
            mode='lines',
            line=dict(color='grey', width=1),
            hoverinfo='none'
        )

        node_traces = []
        for group, color in colordict.items():
            nodes_in_group = [n for n in g.nodes() if g.nodes[n]['group'] == group]
            if not nodes_in_group:
                continue
            node_traces.append(go.Scatter3d(
                x=[pos[n][0] for n in nodes_in_group],
                y=[pos[n][1] for n in nodes_in_group],
                z=[pos[n][2] for n in nodes_in_group],
                mode='markers+text',
                name=group,
                text=[g.nodes[n]['name'] for n in nodes_in_group],
                textposition='top center',
                marker=dict(size=6, color=color),
                hoverinfo='text'
            ))

        fig = go.Figure(data=[edge_trace, *node_traces])
        fig.update_layout(
            showlegend=True,
            scene=dict(
                xaxis=dict(showbackground=False),
                yaxis=dict(showbackground=False),
                zaxis=dict(showbackground=False)
            ),
            margin=dict(l=0, r=0, t=0, b=0)
        )
        fig.show()

def delete_node(g, node_names):
    indices = [n for n in g.nodes if g.nodes[n]['name'] in node_names]
    g_deleted = g.copy()
    for n in indices:
        g_deleted.remove_node(n)
        print(f"Deleted node: {n}")
    return g_deleted
    
        
#plot_3d_graph(g_signal_noise)


In [ ]:
p2last = run_graph("P2 Last 10", threshold = .5, show_plots=False)
p2removed = delete_node(p2last, ["C4", "C3", "Cz"])
plot_3d_graph(p2removed)

In [ ]:
nodes = np.array([pos[v] for v in g_signal_noise])
edges = np.array([(pos[u], pos[v]) for u, v in g_signal_noise.edges()])
fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
ax.clear()
ax.scatter(*nodes.T, alpha=0.2, s=100, color="blue")
for vizedge in edges:
    ax.plot(*vizedge.T, color="gray")
ax.grid(False)
ax.set_axis_off()


def _frame_update(index):
    ax.view_init(index * 0.2, index * 0.5)



fig.tight_layout()

In [ ]:
print(p1.ch_names[26])

In [ ]:
first_corrected, first_higham = redraw("P1 First 10")
last_corrected, last_higham = redraw("P1 Last 10")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

g_first_10 = build_corr_graph(p1_first_10, first_higham, first_corrected, threshold=0.2)
g_last_10 = build_corr_graph(p1_last_10, last_higham, last_corrected, threshold=0.2)

# shared layout so the two snapshots are visually comparable
seed = np.random.random_integers(0,1000)

for ax, g, title in [(ax1, g_first_10, "first 10"), (ax2, g_last_10, "last 10")]:
    node_colors = [g.nodes[n]['color'] for n in g.nodes]
    pos = nx.forceatlas2_layout(g, linlog=True, seed=int(seed))
    nx.draw(g, pos=pos, node_color=node_colors, with_labels=True, ax=ax)
    ax.set_title(title)


In [ ]:
p2_first_corrected, p2_first_higham = redraw("P2 First 10")
p2_last_corrected, p2_last_higham = redraw("P2 Last 10")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

g_first_10 = build_corr_graph(p2_first_10, p2_first_higham, p2_first_corrected, threshold=0.2)
g_last_10 = build_corr_graph(p2_last_10, p2_last_higham, p2_last_corrected, threshold=0.2)

pos = nx.forceatlas2_layout(g_first_10, linlog=True, seed=10)

for ax, g, title in [(ax1, g_first_10, "first 10"), (ax2, g_last_10, "last 10")]:
    node_colors = [g.nodes[n]['color'] for n in g.nodes]
    nx.draw(g, pos=pos, node_color=node_colors, with_labels=True, ax=ax)
    ax.set_title(title)


In [ ]:
dim = 32
rand_eigs = np.random.rand(32)
norm = np.sum(rand_eigs)
rand_eigs = [i/norm*32 for i in rand_eigs]
print(np.sum(rand_eigs))

rand_corr= stats.random_correlation.rvs(rand_eigs)

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(10,10)
im1 = ax.imshow(rand_corr, cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)
ax.set_title('Per-pair optimal lag (s)')

In [ ]:
initialize_graph(p1)
g_rand = build_corr_graph(p1,mask = rand_corr,weights = rand_corr, threshold = .1)
nx.draw(g_rand, pos = nx.forceatlas2_layout(g_rand),node_color = [g_rand.nodes[n]['color'] for n in g_rand.nodes], with_labels = True)

In [ ]:
plot_all_channels_by_trial(p1, 0)
plot_all_channels_by_trial(p1,-1)